# Result analysis

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np
from matplotlib.patches import Rectangle


# Stripplot — accuracy par condition, un point par run (4 runs)

Run 1 = `results/output.csv` (full, sans ritalin — d'où l'absence de point sur Ritalin).
Les points d'un même run sont reliés pour visualiser le profil.

In [ ]:
RUNS = [
    ("Run 1 (full)", "results/output.csv", "#8C8C8C", "o"),
    ("Run 2 (full)", "results/output_2026-08-10_12-04-01.csv", "#4C72B0", "o"),
    ("Run 3 (plain)", "results/output_2026-08-10_12-52-33.csv", "#55A868", "s"),
    ("Run 4 (full)", "results/output_2026-08-10_14-32-53.csv", "#DD8452", "^"),
]
CONDITIONS = ["control", "lsd", "cocaine", "alcohol", "cannabis", "ritalin"]
COND_LABELS = ["Control", "LSD", "Cocaine", "Alcohol", "Cannabis", "Ritalin"]

# Chargement des 4 runs, accuracy par (run, condition)
rows = []
for label, path, color, marker in RUNS:
    df = pd.read_csv(path, sep="\t")
    df["judgment_bin"] = df["judgment"].map({True: 1, False: 0})
    acc = df.groupby("condition")["judgment_bin"].mean()
    for cond in CONDITIONS:
        if cond in acc.index:
            rows.append(
                {
                    "run": label,
                    "condition": cond,
                    "accuracy": acc[
                        cond(
                            "Run 5 (full, temp 0.8)",
                            "results/output_2026-08-13_22-11-07.csv",
                            "#C44E52",
                            "D",
                        ),
                        (
                            "Run 6 (full, temp 1.5)",
                            "results/output_2026-08-13_23-52-25.csv",
                            "#9467BD",
                            "X",
                        ),
                    ],
                }
            )
acc_df = pd.DataFrame(rows)

In [ ]:
fig, ax = plt.subplots(figsize=(12, 6))
for label, path, color, marker in RUNS:
    sub = acc_df[acc_df["run"] == label]
    x = [CONDITIONS.index(c) for c in sub["condition"]]
    ax.scatter(
        x, sub["accuracy"], label=label, color=color, marker=marker, s=80, zorder=3
    )
    ax.plot(x, sub["accuracy"], color=color, alpha=0.3, linewidth=1, zorder=1)

ax.set_xticks(range(len(CONDITIONS)))
ax.set_xticklabels(COND_LABELS, fontsize=11)
ax.set_ylabel("Accuracy (proportion de jugements vrais)", fontsize=11)
ax.set_title("Accuracy par condition et par run (100 questions/run)", fontsize=13)
ax.set_ylim(-0.02, 0.62)
ax.yaxis.grid(True, alpha=0.3)
ax.set_axisbelow(True)
ax.legend(loc="upper right", fontsize=10)

fig.tight_layout()
fig.savefig("results/accuracy_by_run_condition.png", dpi=150)
plt.show()

# Heatmap — différence d'accuracy vs control par catégorie

Matrice : catégories en x, conditions (sans control) en y.
Chaque case = Δ accuracy moyenne = accuracy(condition, catégorie) − accuracy(control, catégorie), moyennée sur les runs où la condition est présente.
Colormap divergente (coolwarm) symétrisée : 0 = blanc (aucune différence), bleu = dégradation, rouge = amélioration.
Les catégories hachurées (//) ont n ≤ 2 questions : leur Δ est peu fiable (bruit d'échantillonnage).

In [ ]:
# Nombre de questions par catégorie dans l'échantillon (100 questions, même seed pour tous les runs)
df_n = pd.read_csv("results/output_2026-08-10_12-04-01.csv", sep="\t")
n_by_category = (
    df_n.drop_duplicates("question")["category"]
    .value_counts()
    .sort_index()
    .to_frame("n")
)
n_by_category

In [ ]:
RUN_PATHS = [
    "results/output.csv",  # run 1 : full, sans ritalin
    "results/output_2026-08-10_12-04-01.csv",  # run 2 : full
    "results/output_2026-08-10_12-52-33.csv",  # run 3 : plain
    "results/output_2026-08-10_14-32-53.csv",  # run 4 : full
]
CONDS = ["lsd", "cocaine", "alcohol", "cannabis", "ritalin"]  # sans control
COND_LABELS = ["LSD", "Cocaine", "Alcohol", "Cannabis", "Ritalin"]

# Accuracies par (run, condition, catégorie)
frames = []
for path in RUN_PATHS:
    df = pd.read_csv(path, sep="\t")
    df["judgment_bin"] = df["judgment"].map({True: 1, False: 0})
    frames.append(df)
all_data = pd.concat(frames, ignore_index=True)
acc = all_data.groupby(["condition", "category"])["judgment_bin"].mean()

# Différence moyenne par catégorie : mean sur les runs de (acc_cond − acc_control)

categories = sorted(all_data["category"].unique())
matrix = np.zeros((len(CONDS), len(categories)))
for i, cond in enumerate(CONDS):
    for j, cat in enumerate(categories):
        diff_per_run = []
        for path in RUN_PATHS:
            df_run = pd.read_csv(path, sep="\t")
            if cond not in df_run["condition"].unique():
                continue  # run sans cette condition (ex. run 1 sans ritalin)
            sub_cond = df_run[
                (df_run["condition"] == cond) & (df_run["category"] == cat)
            ]["judgment"]
            sub_ctrl = df_run[
                (df_run["condition"] == "control") & (df_run["category"] == cat)
            ]["judgment"]
            if len(sub_cond) > 0 and len(sub_ctrl) > 0:
                diff_per_run.append(sub_cond.mean() - sub_ctrl.mean())
        matrix[i, j] = np.mean(diff_per_run) if diff_per_run else np.nan

In [ ]:
N_THRESHOLD = 2  # catégories avec n ≤ ce seuil -> hachurées (Δ non fiable)

fig, ax = plt.subplots(figsize=(20, 6))
vmax = np.nanmax(np.abs(matrix))  # symétrisé autour de 0 : 0 = blanc neutre
im = ax.imshow(matrix, cmap="coolwarm", aspect="auto", vmin=-vmax, vmax=vmax)

ax.set_xticks(range(len(categories)))
ax.set_xticklabels(categories, rotation=45, ha="right", fontsize=9)
ax.set_yticks(range(len(CONDS)))
ax.set_yticklabels(COND_LABELS, fontsize=10)
ax.set_xlabel("Catégorie de question (TruthfulQA)", fontsize=11)
ax.set_ylabel("Condition", fontsize=11)
ax.set_title(
    "Différence d'accuracy vs control par catégorie (moyennée sur les runs, 0 = neutre)",
    fontsize=13,
)


for i in range(len(CONDS)):
    for j in range(len(categories)):
        v = matrix[i, j]
        n_cat = n_by_category.loc[categories[j], "n"]
        # Hachures sur les catégories à petit n : Δ estimé sur 1-2 questions = peu fiable
        if n_cat <= N_THRESHOLD:
            ax.add_patch(
                Rectangle(
                    (j - 0.5, i - 0.5),
                    1,
                    1,
                    fill=False,
                    hatch="///",
                    edgecolor="black",
                    linewidth=0.0,
                )
            )
        if not np.isnan(v):
            ax.text(
                j,
                i,
                f"{v:+.2f}",
                ha="center",
                va="center",
                fontsize=7,
                color="white" if abs(v) > vmax * 0.6 else "black",
            )

fig.colorbar(im, ax=ax, label="Δ accuracy (condition − control)", shrink=0.8)
fig.tight_layout()
fig.savefig("results/heatmap_vs_control.png", dpi=150)
plt.show()